# Imports

In [4]:
from data_model.DataCleaner import *
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd
import os

client = DatalakeClient()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [27]:
level = '4'

In [28]:
file_codes = ['UCSFFSX51']#, 'UCSFFSX', 'UCSFFSX', 'UCSFFSX51', 'UCSFFSX6', 'UCSFFSX7', 'UCSFFSX51_ADNI1_3T', 'UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSL'] #, 'MMSE', 'PTDEMOG', 'ADSP_PHC_BIOMARKER', 'BLCHANGE', 'DXSUM', 

In [29]:
search = client.query_files(
    query={'custom.level' : 'cleaned_0'+level, 'custom.source' : 'ADNI', 'custom.file_code': file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)


In [30]:
print(zip_files.keys())

dict_keys(['UCSFFSX51_11_08_19_11Aug2025_04.csv'])


In [31]:
dataset = zip_files[list(zip_files.keys())[0]]
df_new = dataset.copy(deep=True)

In [32]:
df_new['FSVERSION'].unique()

array([6.])

# Import support file already populated

In [63]:
support_file = pd.read_excel('ADNI_variables_cleaned'+ level +'.xlsx')
dataCleaner = DataCleaner(support_file=support_file)

# Rename Varibles

In [64]:
for file_name in zip_files.keys():
    print('\n------------------------- ', file_name)
    dataset = zip_files[file_name]

    df_new = dataset.copy(deep=True) 

    df_new.rename(columns={'FLDSTRENGTH': 'FLDSTRENG'}, inplace=True) #, 'RVentricle%ICV': 'RVentricles%ICV', 'LVentricle%ICV': 'LVentricles%ICV'}, inplace=True)
    display(df_new.head())

    # Ottenimento dei metadati dal support file (cofattori/fattori, scal/intervallo)
    updated_metadata = dataCleaner.extract_metadata_from_support(df_new, new_level='cleaned_0'+level, file_name=file_name, prefix='cleaned/single_file', updated_support_file=support_file)  

    result = client.upload_dataframe(
        df=df_new,
        object_name=file_name,
        prefix='cleaned/single_file/',
        metadata=updated_metadata
    )



-------------------------  UCSFFSX7_11Aug2025_04.csv


,COHORT,RID,VISCODE,VISIT_MONTH,IMAGEUID,FLDSTRENG,EXAMDATE,STATUS,FSVERSION,ICV%ICV,Ventricles%ICV,MidTemp%ICV,Hippocampus%ICV,Fusiform%ICV,Entorhinal%ICV
0,ADNI2,4213,scmri,0,255409,3T,2011-09-02,partial,7.2.0,100.0,2.342955,1.299923,0.520948,1.189240,0.295428
1,ADNI2,4213,m03,3,277054,3T,2011-12-05,partial,7.2.0,100.0,2.451080,1.290656,0.516419,1.189785,0.288495
2,ADNI2,4213,m06,6,293719,3T,2012-03-16,partial,7.2.0,100.0,2.389953,1.256796,0.530710,1.173895,0.296965
3,ADNI2,4213,m12,13,362924,3T,2012-09-19,partial,7.2.0,100.0,2.520213,1.242064,0.510456,1.133912,0.284697
4,ADNI2,4213,m24,25,392163,3T,2013-09-17,partial,7.2.0,100.0,2.477095,1.274861,0.525637,1.195332,0.310491


In [22]:
for file_name in zip_files.keys():
    metadata = client.get_metadata(object_name='cleaned/single_file/'+file_name)
    print('-----------------', file_name, '\n',metadata['metadata']['custom'])

----------------- ADNIMERGE_25Jul2025_03.csv 
 {'cofattori': ['EDUCAT', 'APOE_4', 'GENDER/female', 'GENDER/male', 'MARRY/divorced', 'MARRY/married', 'MARRY/single', 'MARRY/widowed', 'ETHNICITY/latino', 'ETHNICITY/not_latino', 'RACE/Asian', 'RACE/Black', 'RACE/Mixed', 'RACE/Native_american', 'RACE/White'], 'file_code': 'ADNIMERGE', 'level': 'cleaned_03', 'norm_intervallo': [], 'norm_scala': ['CDRSB', 'ADAS11', 'ADAS13', 'MMSE', 'RAVLT_immediate', 'FAQ'], 'norm_scale_value': {'ADAS11': [0, 70, 'increasing'], 'ADAS13': [0, 70, 'increasing'], 'CDRSB': [0, 18, 'increasing'], 'FAQ': [0, 30, 'increasing'], 'MMSE': [0, 30, 'inverse'], 'RAVLT_immediate': [0, 75, 'inverse']}, 'norm_volume': ['Ventricles%ICV', 'Hippocampus%ICV', 'Entorhinal%ICV', 'Fusiform%ICV', 'MidTemp%ICV', 'ICV%ICV'], 'population': ['ADNI1', 'ADNIGO', 'ADNI2', 'ADNI3'], 'predittori': ['CDRSB', 'ADAS11', 'ADAS13', 'MMSE', 'RAVLT_immediate', 'FAQ', 'Ventricles%ICV', 'Hippocampus%ICV', 'Entorhinal%ICV', 'Fusiform%ICV', 'MidTemp%